In [1]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sagemaker import get_execution_role

session = boto3.session.Session()
region = session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
sm_session = sagemaker.Session()
bucket = sm_session.default_bucket()

print(f"Region: {region}")
print(f"Account ID: {account_id}")
print(f"SageMaker SDK version: {sagemaker.__version__}")
print(f"Default S3 bucket: {bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/santi/Library/Application Support/sagemaker/config.yaml
Region: us-east-1
Account ID: 961538614332
SageMaker SDK version: 2.257.1
Default S3 bucket: sagemaker-us-east-1-961538614332


In [ ]:
import os

# Save dataset locally first
train_file = '03-sagemaker/train.csv'
test_file = '03-sagemaker/test.csv'

# Rebuild train/test split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Reload dataset if not in memory
url = "https://raw.githubusercontent.com/jmnote/z-labs-src/master/NSL-KDD/KDDTrain%2B.csv"
df = pd.read_csv(url)
df['is_attack'] = (df['label'] != 'normal').astype(int)

numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_features = [f for f in numeric_features if f not in ['is_attack', 'difficulty']]

X = df[numeric_features].fillna(0)
y = df['is_attack']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# Save as CSV with label in first column (SageMaker expected format)
train_df = pd.DataFrame(X_train, columns=numeric_features)
train_df.insert(0, 'is_attack', y_train.values)

test_df = pd.DataFrame(X_test, columns=numeric_features)
test_df.insert(0, 'is_attack', y_test.values)

os.makedirs('03-sagemaker', exist_ok=True)
train_df.to_csv(train_file, index=False, header=False)
test_df.to_csv(test_file, index=False, header=False)

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")
print("CSV files saved.")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:13                                                                                   │
│                                                                                                  │
│   10                                                                                             │
│   11 # Reload dataset if not in memory                                                           │
│   12 url = "https://raw.githubusercontent.com/jmnote/z-labs-src/master/NSL-KDD/KDDTrain+.csv"    │
│ ❱ 13 df = pd.read_csv(url)                                                                       │
│   14 df['is_attack'] = (df['label'] != 'normal').astype(int)                                     │
│   15                                                                                             │
│   16 numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()                   │
│                                                                                                  │
│ /Users/santi/Documents/sources/detection-engineering-prep/venv/lib/python3.14/site-packages/pand │
│ as/io/parsers/readers.py:873 in read_csv                                                         │
│                                                                                                  │
│    870 │   )                                                                                     │
│    871 │   kwds.update(kwds_defaults)                                                            │
│    872 │                                                                                         │
│ ❱  873 │   return _read(filepath_or_buffer, kwds)                                                │
│    874                                                                                           │
│    875                                                                                           │
│    876 @overload                                                                                 │
│                                                                                                  │
│ /Users/santi/Documents/sources/detection-engineering-prep/venv/lib/python3.14/site-packages/pand │
│ as/io/parsers/readers.py:300 in _read                                                            │
│                                                                                                  │
│    297 │   _validate_names(kwds.get("names", None))                                              │
│    298 │                                                                                         │
│    299 │   # Create the parser.                                                                  │
│ ❱  300 │   parser = TextFileReader(filepath_or_buffer, **kwds)                                   │
│    301 │                                                                                         │
│    302 │   if chunksize or iterator:                                                             │
│    303 │   │   return parser                                                                     │
│                                                                                                  │
│ /Users/santi/Documents/sources/detection-engineering-prep/venv/lib/python3.14/site-packages/pand │
│ as/io/parsers/readers.py:1645 in __init__                                                        │
│                                                                                                  │
│   1642 │   │   │   self.options["has_index_names"] = kwds["has_index_names"]                     │
│   1643 │   │                                                                                     │
│   1644 │   │   self.handles: IOHandles | None = None                                             │
│ ❱ 1645 │   │   self._engine = self._make_engine(f, self.eng